In [1]:
import os
import sys
import time
# sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS


from src.config import HR_DOCS_DIR, FAISS_INDEX_DIR, CHUNK_SIZE, CHUNK_OVERLAP, EMBEDDING_MODEL,OPENAI_API_KEY



def load_documents():
    loader = DirectoryLoader(HR_DOCS_DIR, glob="**/*.txt", show_progress=True, loader_cls=TextLoader)
    documents = loader.load()
    print(f"  Loaded {len(documents)} documents")
    return documents

def chunk_documents(docs):
    """Split documents into chunks."""
    print(f"\n  Chunking with size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""],
        length_function=len,
    )
    chunks = splitter.split_documents(docs)
    print(f"  Created {len(chunks)} chunks")
    
    for i,chunk in enumerate(chunks):
        source_file=os.path.basename(chunk.metadata.get("source","unknown"))
        chunk.metadata["source_file"]=source_file
        chunk.metadata["doc_name"]="HR"
        chunk.metadata["chunk_id"]=i

    print(f"  Created {len(chunks)} chunks from {len(docs)} documents")

    print(f"\n  Sample chunk (#{0}):")
    print(f"  Source: {chunks[0].metadata['doc_name']}")
    print(f"  Content: {chunks[0].page_content[:200]}...")

    return chunks

def embed_and_save(chunks):

    print(f"\n Embedding {len(chunks)} chunks using {EMBEDDING_MODEL} model ....")
    embeddings=OpenAIEmbeddings(model=EMBEDDING_MODEL, openai_api_key=OPENAI_API_KEY)

    start=time.time()
    vectorstore=FAISS.from_documents(chunks, embeddings)
    elapsed=time.time()-start

    print(f"Embedded in elapsed time: {elapsed:.2f} seconds")

    os.makedirs(os.path.dirname(FAISS_INDEX_DIR), exist_ok=True)
    vectorstore.save_local(FAISS_INDEX_DIR)
    print(f"Saved FAISS index to {FAISS_INDEX_DIR}")

    return vectorstore

def run_ingest():
    print("Starting HR document ingestion...")
    documents = load_documents()
    chunks = chunk_documents(documents)
    vectorstore = embed_and_save(chunks)
    print(f"  DONE! {len(chunks)} chunks indexed and saved.")
    print("Ingestion complete.")

    return vectorstore




: 

In [ ]:
import os
import sys
# sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

from src.config import FAISS_INDEX_DIR, EMBEDDING_MODEL, OPENAI_API_KEY,TOP_K, SYSTEM_PROMPT, LLM_MODEL, TEMPERATURE

class simple_retriever:
    def __init__(self):
        self.vectorstore = None
        self._load_index()


    def _load_index(self):
        if not os.path.exists(FAISS_INDEX_DIR):
            raise FileNotFoundError(f"FAISS index not found at {FAISS_INDEX_DIR}. Please run the ingest script first.")
        
        print(f"Loading FAISS index from {FAISS_INDEX_DIR}...")
        embeddings=OpenAIEmbeddings(model=EMBEDDING_MODEL, openai_api_key=OPENAI_API_KEY)
        self.vectorstore=FAISS.load_local(FAISS_INDEX_DIR, embeddings,allow_dangerous_deserialization=True)

        print("FAISS index loaded successfully.")

    def retrieve(self, query:str,top_k:int=TOP_K):
        results = self.vectorstore.similarity_search_with_score(query, k=top_k)
        return results
    
    def get_context(self,query:str, top_k:int=TOP_K)->str:

        results=self.retrieve(query,top_k)
        
        context_parts=[]
        
        for i ,(doc,score) in enumerate(results,1):

            source = doc.metadata.get("doc_name", "Unknown")
            dept = doc.metadata.get("dept", "Unknown")
            context_parts.append(
                f"[Source {i}: {source} | Dept: {dept} | Relevance: {score:.3f}]\n"
                f"{doc.page_content}"
            )
        return "\n\n---\n\n".join(context_parts)
    
    def get_sources(self,query:str,top_k:int=TOP_K)->list:
        results=self.retrieve(query,top_k)
        sources=[]
        for doc, score in results:
            sources.append({
                "source": doc.metadata.get("doc_name", "Unknown"),
                "dept": doc.metadata.get("dept", "Unknown"),
                "score": float(score),
                "preview": doc.page_content[:150] + "...",
            })
        return sources
    

if __name__ == "__main__":
    retriever = simple_retriever()

    test_query = "What is the annual leave entitlement?"
    print(f"\nQuery: {test_query}\n")

    sources = retriever.get_sources(test_query)
    for i, src in enumerate(sources, 1):
        print(f"  {i}. [{src['dept']}] {src['source']} (score: {src['score']:.3f})")
        print(f"     {src['preview']}\n")